# 3강 실습: 규칙 기반 검사 — 자로 재듯이 검사하기

**사용법**: 위에서부터 코드 칸을 클릭하고 `Shift + Enter`.

**오늘 할 것**
- 0부. 2강 연습문제 3번 복습: 제품이 사진 밖으로 나가면? → "검사 불가"
- 1부. 연습용 부품 사진 만들기 (크기·찌그러짐·깨짐 불량)
- 2부. 엣지와 컨투어, 그리고 측정 (면적·지름·진원도·볼록도, 픽셀 → mm)
- 3부. 🏭 미니 프로젝트 ①: 도면 규격으로 판정하는 검사기 (불량 이유까지)
- 4부. 🏭 미니 프로젝트 ②: 트레이 위 부품 여러 개 세고 하나씩 판정

> `#` 뒤 글은 설명(주석)입니다. 그래프 제목은 한글이 깨지지 않도록 영어로 씁니다.

## 준비: 라이브러리 불러오기

In [ ]:
import numpy as np                 # 숫자 계산
import cv2                         # 이미지 처리
import matplotlib.pyplot as plt    # 그림 그리기
import pandas as pd                # 표(엑셀 같은 표) 만들기 도구 — 결과를 보기 좋게 정리할 때 씀

print("준비 완료!")

---
# 0부. 2강 연습문제 3번 복습: "검사 불가"

2강에서 `make_part_photo(dx=150)` 로 제품을 오른쪽 끝으로 보내면 오류가 났습니다.
이유: 제품 중심 ±100 픽셀을 자르려는데, **사진 오른쪽 끝(480)을 넘어가서** 잘린 크기가 200×200이 아니게 되기 때문입니다.

작은 예로 확인해 봅니다. 가로 10칸짜리 줄에서 7~13번 칸을 자르려고 하면?

In [ ]:
row = np.arange(10)          # 0, 1, 2, ..., 9 (10칸)
piece = row[7:13]            # 7번부터 12번까지 6칸을 원했지만…
print("원한 칸 수: 6 / 실제로 잘린 칸 수:", len(piece), "→", piece)
print("→ 끝을 넘어가면 오류 대신 '조용히' 짧게 잘립니다. 그래서 다음 계산에서 크기가 안 맞아 오류가 납니다.")

**해결**: 자르기 전에 **"제품이 사진 가장자리에 닿았나?"** 를 먼저 검사하고, 닿았으면 OK/NG 대신 **"검사 불가"** 로 처리합니다.

`cv2.boundingRect(외곽선)` 은 덩어리를 감싸는 네모의 `(x, y, 가로, 세로)` 를 알려 줍니다. 이 네모가 사진 끝에 닿으면 검사 불가입니다.

In [ ]:
def touches_border(contour, image_shape, margin=2):
    h_img, w_img = image_shape[:2]                     # 사진의 세로, 가로 크기
    x, y, w, h = cv2.boundingRect(contour)             # 덩어리를 감싸는 네모
    return (x <= margin or y <= margin or              # 왼쪽 또는 위쪽 끝에 닿음
            x + w >= w_img - margin or                 # 오른쪽 끝에 닿음
            y + h >= h_img - margin)                   # 아래쪽 끝에 닿음

print("함수 준비 완료 (아래 3부에서 사용합니다)")

---
# 1부. 연습용 부품 사진 만들기

2강과 같은 금속 링이지만, 이번에는 **치수 불량**을 만들 수 있습니다.
이 사진에서 **1픽셀 = 0.1 mm** 라고 정합니다. (반지름 90픽셀 → 지름 180픽셀 → **18.0 mm**)

| 설정 | 뜻 | 양품 값 |
|---|---|---|
| `outer` | 바깥 반지름 (픽셀) | 90 → 외경 18.0 mm |
| `hole` | 구멍 반지름 (픽셀) | 30 → 내경 6.0 mm |
| `squash` | 세로로 찌그러진 정도 (1.0 = 완전한 원) | 1.0 |
| `chip` | 가장자리 깨짐 | False |
| `dx`, `dy`, `light`, `dust` | 위치·조명·먼지 (2강과 같음) | 0 |

> 이 칸은 "사진 만드는 공장"입니다. 실행만 해도 됩니다.

In [ ]:
rng = np.random.default_rng(seed=3)    # 난수 고정 → 매번 같은 결과
H, W = 360, 480                        # 사진 크기 (세로, 가로)
MM_PER_PIXEL = 0.1                     # 캘리브레이션: 1픽셀 = 0.1 mm

def make_part(dx=0, dy=0, outer=90, hole=30, squash=1.0, chip=False, light=0, noise=4, dust=0.0):
    img = np.full((H, W), 60, dtype=np.float32)                 # 어두운 컨베이어 배경
    img += rng.normal(0, 4, (H, 1)) * np.ones((1, W))           # 벨트 줄무늬
    c = (W // 2 + dx, H // 2 + dy)                              # 제품 중심
    cv2.ellipse(img, c, (outer, int(round(outer * squash))), 0, 0, 360, 185, -1)  # 바깥 원(타원)
    cv2.circle(img, c, hole, 60, -1)                            # 구멍
    if chip:                                                    # 깨짐: 가장자리를 동그랗게 파냄
        cv2.circle(img, (c[0] + int(outer * 0.92), c[1] - int(outer * 0.39)), 18, 60, -1)
    img += light                                                # 조명 변화
    img += rng.normal(0, noise, img.shape)                      # 센서 잡음
    img = np.clip(img, 0, 255).astype(np.uint8)                 # 0~255로 맞추기
    if dust > 0:                                                # 먼지 점
        r = rng.random(img.shape)
        img[r < dust / 2] = 0
        img[r > 1 - dust / 2] = 255
    return img

examples = {
    "Good":             make_part(),
    "Hole too big":     make_part(hole=33),
    "Outer too small":  make_part(outer=86),
    "Oval (squashed)":  make_part(squash=0.94),
    "Chipped edge":     make_part(chip=True),
    "Off the edge":     make_part(dx=170),
}
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, im) in zip(axes.ravel(), examples.items()):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255)
    ax.set_title(name)
    ax.axis("off")
plt.show()
print("→ 구멍 크기, 바깥 크기, 찌그러짐은 눈으로 구분하기 어렵습니다. 그래서 '숫자로 재는' 검사가 필요합니다.")

---
# 2부. 엣지와 컨투어, 그리고 측정

## ① 엣지: 밝기가 갑자기 바뀌는 곳

`cv2.Canny(사진, 낮은기준, 높은기준)` — 밝기 변화가 '높은기준'보다 크면 확실한 엣지, '낮은기준'~'높은기준' 사이면 확실한 엣지와 이어질 때만 엣지로 인정합니다.

✏️ 50, 150을 바꿔 보세요. 낮추면 잡음까지 선으로 잡히고, 높이면 선이 끊깁니다.

In [ ]:
good = examples["Good"]
blur = cv2.GaussianBlur(good, (5, 5), 0)       # 2강: 엣지 전에 잡음 줄이기
edges = cv2.Canny(blur, 50, 150)               # ✏️ 엣지 찾기

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].imshow(good, cmap="gray", vmin=0, vmax=255); axes[0].set_title("Photo")
axes[1].imshow(edges, cmap="gray");                  axes[1].set_title("Canny edges")
for ax in axes:
    ax.axis("off")
plt.show()

## ② 컨투어: 흰 덩어리의 테두리 좌표 목록

순서: 이진화(2강 Otsu) → `cv2.findContours` 로 테두리 찾기

- `cv2.RETR_CCOMP` : "바깥 테두리"와 "그 안의 구멍 테두리"를 **부모-자식**으로 구분해서 찾아 줍니다.
- `hierarchy[0][k][3]` : k번째 테두리의 **부모 번호**. 부모가 없으면 -1 (= 가장 바깥 테두리)
- `cv2.CHAIN_APPROX_NONE` : 테두리 점을 생략하지 않고 전부 저장 (정밀 측정용)

In [ ]:
clean = cv2.medianBlur(good, 5)                                          # 먼지·잡음 제거
_, mask = cv2.threshold(clean, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)   # 제품=흰색, 배경=검정
contours, hierarchy = cv2.findContours(mask, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)

print("찾은 테두리 개수:", len(contours))
for k, c in enumerate(contours):                         # k = 번호, c = 테두리 좌표들
    parent = hierarchy[0][k][3]                          # 이 테두리의 부모 번호
    kind = "바깥 테두리" if parent == -1 else f"{parent}번의 안쪽(구멍)"
    print(f"  {k}번: 점 {len(c)}개, 면적 {cv2.contourArea(c):8.0f} 픽셀 → {kind}")

view = cv2.cvtColor(good, cv2.COLOR_GRAY2RGB)                   # 컬러로 바꿔서 색깔 선 그리기
for k, c in enumerate(contours):
    color = (255, 0, 0) if hierarchy[0][k][3] == -1 else (0, 200, 255)   # 바깥=빨강, 구멍=하늘색
    cv2.drawContours(view, [c], -1, color, 2)                   # 테두리 그리기
plt.figure(figsize=(6, 4.5))
plt.imshow(view)
plt.title("Contours: red = outer, blue = hole")
plt.axis("off")
plt.show()

## ③ 측정: 지름, 진원도, 볼록도

- `cv2.fitEllipse(테두리)` → 테두리에 가장 잘 맞는 타원. 결과 중 `(지름1, 지름2)` 를 씁니다.
  - **지름** = 두 지름의 평균 × 0.1 mm
  - **진원도** = 짧은 지름 ÷ 긴 지름 (1.0 = 완전한 원)
- `cv2.convexHull(테두리)` → 고무줄로 감싼 모양
  - **볼록도** = 실제 면적 ÷ 고무줄 면적 (깨진 곳이 있으면 작아짐)

In [ ]:
def find_outer_and_hole(photo):
    clean = cv2.medianBlur(photo, 5)
    _, mask = cv2.threshold(clean, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, hierarchy = cv2.findContours(mask, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
    outer_idx = max(range(len(contours)), key=lambda k: cv2.contourArea(contours[k]))  # 가장 큰 = 제품 바깥
    holes = [contours[k] for k in range(len(contours)) if hierarchy[0][k][3] == outer_idx]  # 그 자식들 = 구멍
    hole = max(holes, key=cv2.contourArea) if holes else None    # 구멍이 여러 개면 가장 큰 것
    return contours[outer_idx], hole

def diameter_mm(contour):
    (_, _), (d1, d2), _ = cv2.fitEllipse(contour)      # 타원 맞추기 → 두 지름(픽셀)
    return (d1 + d2) / 2 * MM_PER_PIXEL                # 평균 지름 → mm

def roundness(contour):
    (_, _), (d1, d2), _ = cv2.fitEllipse(contour)
    return min(d1, d2) / max(d1, d2)                   # 짧은 지름 ÷ 긴 지름

def solidity(contour):
    hull = cv2.convexHull(contour)                     # 고무줄로 감싼 모양
    return cv2.contourArea(contour) / cv2.contourArea(hull)

outer, hole = find_outer_and_hole(good)
print(f"외경   : {diameter_mm(outer):.2f} mm   (만들 때 정한 값 18.0)")
print(f"내경   : {diameter_mm(hole):.2f} mm   (만들 때 정한 값 6.0)")
print(f"진원도 : {roundness(outer):.3f}")
print(f"볼록도 : {solidity(outer):.3f}")

측정값이 정확히 18.00, 6.00이 아니라 조금 다르죠? 테두리가 픽셀 단위로 계단처럼 생겼고, 잡음과 블러 때문에 생기는 **측정 오차**입니다.
그래서 공차(± 0.3 mm)를 측정 오차보다 충분히 크게 잡아야 합니다. 실제 현장에서는 **같은 부품을 여러 번 찍어서 측정값이 얼마나 흔들리는지**를 먼저 확인합니다.

### 깨진 부품은 고무줄(볼록 껍질)과 어떻게 다를까?

In [ ]:
chipped = examples["Chipped edge"]
outer_c, _ = find_outer_and_hole(chipped)
hull = cv2.convexHull(outer_c)

view = cv2.cvtColor(chipped, cv2.COLOR_GRAY2RGB)
cv2.drawContours(view, [hull], -1, (0, 255, 0), 2)       # 초록 = 고무줄
cv2.drawContours(view, [outer_c], -1, (255, 0, 0), 2)    # 빨강 = 실제 테두리
plt.figure(figsize=(6, 4.5))
plt.imshow(view)
plt.title(f"Chipped: solidity = {solidity(outer_c):.3f}")
plt.axis("off")
plt.show()
print("→ 깨진 곳에서 초록 선(고무줄)과 빨간 선(실제)이 벌어집니다. 그만큼 볼록도가 줄어듭니다.")

---
# 3부. 🏭 미니 프로젝트 ①: 도면 규격 검사기

규격(스펙)을 표처럼 정해 두고, 측정값이 범위를 벗어나면 **NG + 이유**를 남깁니다.

✏️ 규격 숫자를 바꿔서 판정이 어떻게 달라지는지 보세요.

In [ ]:
SPEC = {
    "외경(mm)":  (17.7, 18.3),    # 18.0 ± 0.3
    "내경(mm)":  (5.8, 6.2),      # 6.0 ± 0.2
    "진원도":    (0.97, 1.0),     # 0.97 이상
    "볼록도":    (0.985, 1.0),    # 0.985 이상
}

def inspect_part(photo):
    outer, hole = find_outer_and_hole(photo)
    if touches_border(outer, photo.shape):                       # 0부: 가장자리에 걸쳤나?
        return {"판정": "검사 불가", "이유": "제품이 사진 밖으로 나감"}
    if hole is None:                                             # 구멍이 아예 없음
        return {"판정": "NG", "이유": "구멍 없음"}

    values = {                                                   # 4가지 측정
        "외경(mm)": diameter_mm(outer),
        "내경(mm)": diameter_mm(hole),
        "진원도":   roundness(outer),
        "볼록도":   solidity(outer),
    }
    reasons = []                                                 # 불량 이유를 모을 빈 목록
    for name, value in values.items():
        low, high = SPEC[name]                                   # 이 항목의 허용 범위
        if value < low:
            reasons.append(f"{name} 작음({value:.3f} < {low})")
        elif value > high:
            reasons.append(f"{name} 큼({value:.3f} > {high})")

    result = {k: round(v, 3) for k, v in values.items()}         # 측정값 소수점 3자리로
    result["판정"] = "NG" if reasons else "OK"                    # 이유가 하나라도 있으면 NG
    result["이유"] = ", ".join(reasons)                           # 이유들을 쉼표로 이어 붙임
    return result

print("검사기 준비 완료")

### 시험: 정답을 아는 사진 8장

`pd.DataFrame(목록)` 은 결과를 **엑셀 표처럼** 보여 줍니다.

In [ ]:
tests = [
    ("양품",                    "OK",        make_part()),
    ("양품+이동+밝음+먼지",       "OK",        make_part(dx=30, dy=-20, light=40, dust=0.02)),
    ("양품+어두움",              "OK",        make_part(light=-40)),
    ("구멍 큼",                  "NG",        make_part(hole=33)),
    ("바깥 작음",                "NG",        make_part(outer=86)),
    ("찌그러짐",                 "NG",        make_part(squash=0.94)),
    ("깨짐",                     "NG",        make_part(chip=True)),
    ("사진 밖으로 나감",          "검사 불가",  make_part(dx=170)),
]

rows = []
for name, answer, photo in tests:
    r = inspect_part(photo)                     # 검사!
    r = {"사진": name, "정답": answer, **r}     # 사진 이름·정답을 앞에 붙이기 (** = 사전 풀어 넣기)
    r["맞음"] = "O" if r["판정"] == answer else "X"
    rows.append(r)

table = pd.DataFrame(rows).fillna("")           # 표 만들기 (빈칸은 공백으로)
print("정확도:", (table["맞음"] == "O").sum(), "/", len(table))
table

→ "찌그러짐"과 "깨짐"은 외경도 함께 줄어들어서 이유가 여러 개 나올 수 있습니다. 실제 현장에서도 불량 하나가 여러 측정값에 영향을 줍니다.

### 결과를 그림으로 확인

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, (ax, (name, answer, photo), row) in enumerate(zip(axes.ravel(), tests, rows), start=1):   # i = 1, 2, 3 ...
    view = cv2.cvtColor(photo, cv2.COLOR_GRAY2RGB)
    outer, hole = find_outer_and_hole(photo)
    color = {"OK": (0, 200, 0), "NG": (255, 0, 0)}.get(row["판정"], (255, 200, 0))  # OK 초록, NG 빨강, 검사불가 노랑
    cv2.drawContours(view, [outer], -1, color, 3)
    if hole is not None:
        cv2.drawContours(view, [hole], -1, color, 2)
    ax.imshow(view)
    verdict = {"OK": "OK", "NG": "NG", "검사 불가": "CANNOT INSPECT"}[row["판정"]]
    ax.set_title(f"Test {i}: {verdict}")
    ax.axis("off")
plt.show()

---
# 4부. 🏭 미니 프로젝트 ②: 트레이 위 부품 개수 세기 + 각각 판정

트레이에 부품 **6칸**이 있고, 그중 하나는 비어 있고, 하나는 크기가 작은 불량입니다.
- 부품 개수가 6개가 아니면 → **"수량 부족"** 경고
- 부품마다 외경을 재서 → 규격 밖이면 빨간색

In [ ]:
def make_tray(missing_slot=4, small_slot=1):
    img = np.full((360, 600), 60, dtype=np.float32)              # 트레이 바닥
    slots = [(100 + 200 * col, 90 + 180 * row) for row in range(2) for col in range(3)]  # 2줄 x 3칸 위치
    for i, (x, y) in enumerate(slots):
        if i == missing_slot:                                    # 빈 칸
            continue
        r = 60 if i == small_slot else 65                        # 작은 불량은 반지름 60
        cv2.circle(img, (x, y), r, 185, -1)                      # 부품 바깥
        cv2.circle(img, (x, y), 20, 60, -1)                      # 부품 구멍
    img += rng.normal(0, 4, img.shape)
    return np.clip(img, 0, 255).astype(np.uint8)

tray = make_tray()
plt.figure(figsize=(8, 5))
plt.imshow(tray, cmap="gray", vmin=0, vmax=255)
plt.title("Tray with 6 slots")
plt.axis("off")
plt.show()

In [ ]:
EXPECTED_COUNT = 6                    # 트레이 한 판에 있어야 할 개수
TRAY_SPEC_MM = (12.7, 13.3)           # 이 부품의 외경 규격: 13.0 ± 0.3 mm
MIN_PART_AREA = 2000                  # 이보다 작은 덩어리는 부품이 아님(먼지 등)

clean = cv2.medianBlur(tray, 5)
_, mask = cv2.threshold(clean, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
contours, hierarchy = cv2.findContours(mask, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)

parts = [c for k, c in enumerate(contours)
         if hierarchy[0][k][3] == -1 and cv2.contourArea(c) > MIN_PART_AREA]   # 바깥 테두리 + 충분히 큰 것
print(f"찾은 부품: {len(parts)}개 / 있어야 할 개수: {EXPECTED_COUNT}개",
      "→ ⚠️ 수량 부족!" if len(parts) < EXPECTED_COUNT else "→ 수량 OK")

view = cv2.cvtColor(tray, cv2.COLOR_GRAY2RGB)
parts = sorted(parts, key=lambda c: (cv2.boundingRect(c)[1] // 100, cv2.boundingRect(c)[0]))  # 위→아래, 왼→오 순서로 번호
for n, c in enumerate(parts, start=1):                    # 1번부터 번호 매기기
    d = diameter_mm(c)
    ok = TRAY_SPEC_MM[0] <= d <= TRAY_SPEC_MM[1]          # 규격 안이면 True
    color = (0, 200, 0) if ok else (255, 0, 0)
    cv2.drawContours(view, [c], -1, color, 3)
    x, y, w, h = cv2.boundingRect(c)
    cv2.putText(view, f"#{n} {d:.1f}mm", (x, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)  # 글자 쓰기
    print(f"  #{n}: 외경 {d:.2f} mm → {'OK' if ok else 'NG'}")

plt.figure(figsize=(10, 6))
plt.imshow(view)
plt.title(f"Found {len(parts)} / {EXPECTED_COUNT} parts (green OK, red NG)")
plt.axis("off")
plt.show()

---
## ✅ 3강 정리

- **엣지** = 밝기가 바뀌는 선(그림), **컨투어** = 덩어리 테두리 좌표(계산 가능).
- 컨투어로 **면적, 지름, 진원도, 볼록도**를 재고, **1픽셀 = ? mm** 로 실제 크기로 바꾼다.
- 규격표와 비교해서 **OK / NG + 이유**를 남긴다. 이유 기록이 공정 개선의 출발점.
- 제품이 사진 끝에 걸치면 **"검사 불가"** 로 따로 분류한다.
- 개수 세기, 유무 확인, 치수 측정은 **AI 없이도** 잘 된다.

> 🏭 회사 적용 포인트: 지금 사람이 **자·버니어캘리퍼스·눈으로 세기** 로 하는 검사가 있다면, 오늘 방법으로 자동화할 수 있는 후보입니다.
> 반대로 **모양이 매번 다른 긁힘·얼룩·오염**은 규칙으로 정하기 어렵습니다 → 4강부터 배울 AI의 영역입니다.

## 📝 연습문제

1. `SPEC` 의 외경 공차를 ± 0.1 mm(17.9 ~ 18.1)로 좁혀 보세요. 양품도 NG가 되나요? 측정 오차와 공차의 관계를 생각해 보세요.
2. `make_part(hole=31)` (내경 6.2 mm, 딱 경계값)을 검사해 보세요. 몇 번 실행해도 결과가 같나요? 경계값 근처 제품은 현장에서 어떻게 처리하면 좋을까요?
3. 4부에서 `make_tray(missing_slot=None, small_slot=None)` 로 모두 정상인 트레이를 만들어 확인해 보세요.
4. 생각해 보기: 우리 회사 검사 중 **"재고, 세고, 있는지 확인하는"** 검사는 무엇이 있나요? 그 검사의 규격표를 만들어 본다면?

모르는 부분은 "3강 3부 inspect_part 함수 설명해줘"처럼 물어보세요.